# CSI 300 next-day upward-event classification

A research workflow using daily price/volume features and LightGBM. The target is whether the next trading day's **log return exceeds 0.5%**; class 0 also includes small positive returns. This models one index time series, not stock selection across index constituents.

This public notebook contains rewritten explanations and selected research computations. The original assignment, report, vendor data, authentication, and saved notebook outputs are excluded. Run locally with a CSV matching the README. The default input is explicitly synthetic and demonstrates execution only.

**Historical study, not results from this public run:** 46 candidate features reduced to 3; 869 training and 218 test observations; held-out ROC-AUC 0.5302 versus baseline 0.4891. These numbers come from the earlier local study and are not guarantees for another dataset or dependency version.


## 1. Configuration

Run from this project directory or its `notebooks/` directory. `QUICK_MODE=True` uses a small parameter grid for a smoke run; `False` restores the historical 486-candidate grid. Both modes retain the feature-selection workflow. Output metrics are tagged with the data kind and search mode.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import talib as ta
from lightgbm import LGBMClassifier
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve,
)

cwd = Path.cwd().resolve()
PROJECT_DIR = cwd.parent if cwd.name == 'notebooks' else cwd
if not (PROJECT_DIR / 'requirements.txt').is_file():
    raise RuntimeError('Start Jupyter from projects/csi300-lightgbm or its notebooks directory.')

SEED = 42
QUICK_MODE = True
DATA_SOURCE_KIND = 'synthetic'  # Change to 'licensed-local' for your own authorized data.
DATA_PATH = PROJECT_DIR / 'data' / 'private' / 'synthetic_daily.csv'
WRITE_RESULTS = False
RESULTS_DIR = PROJECT_DIR / 'outputs' / 'private'
np.random.seed(SEED)


## 2. Load a local CSV and construct the target

Required columns: `time, open_price, pre_close_price, high_price, low_price, close_price_nvnf, volume`. Prices must use one consistent adjustment basis, and `pre_close_price` must be the previous trading session's close. All rows must represent the same instrument. This notebook checks structure, numeric validity, date uniqueness and OHLC consistency; it cannot establish data provenance or vendor licensing.


In [ ]:
required = ['time', 'open_price', 'pre_close_price', 'high_price',
            'low_price', 'close_price_nvnf', 'volume']
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'{DATA_PATH.name} is missing. Follow README data setup first.')
df = pd.read_csv(DATA_PATH)
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')
df = df.loc[:, required].copy()
df['time'] = pd.to_datetime(df['time'], errors='raise')
if df['time'].duplicated().any():
    raise ValueError('Use one observation per trading date for one instrument.')
df = df.sort_values('time').reset_index(drop=True)
numeric = required[1:]
df[numeric] = df[numeric].apply(pd.to_numeric, errors='raise')
if not np.isfinite(df[numeric].to_numpy()).all() or (df[numeric] <= 0).any().any():
    raise ValueError('Prices and volume must be finite and strictly positive.')
if (df['high_price'] < df[['open_price', 'close_price_nvnf', 'low_price']].max(axis=1)).any():
    raise ValueError('high_price is inconsistent with OHLC data.')
if (df['low_price'] > df[['open_price', 'close_price_nvnf', 'high_price']].min(axis=1)).any():
    raise ValueError('low_price is inconsistent with OHLC data.')
if not np.allclose(df['pre_close_price'].iloc[1:], df['close_price_nvnf'].iloc[:-1], rtol=1e-5):
    raise ValueError('Previous-close values do not align with the preceding row; check missing sessions/adjustments.')
if len(df) < 750:
    raise ValueError('Provide at least 750 daily rows for this demonstration and its five time-series folds.')
if DATA_SOURCE_KIND not in {'synthetic', 'licensed-local'}:
    raise ValueError('Set DATA_SOURCE_KIND to synthetic or licensed-local.')

df['daily_return'] = np.log(df['close_price_nvnf'] / df['pre_close_price'])
df['next_day_return'] = df['daily_return'].shift(-1)
df = df.dropna(subset=['next_day_return']).copy()
df['label'] = (df['next_day_return'] > 0.005).astype(int)
print({'data_kind': DATA_SOURCE_KIND, 'rows_with_target': len(df),
       'positive_fraction': float(df['label'].mean())})


## 3. Candidate features

The original 46-feature calculations are retained: return, range, volatility, momentum, trend, volume, distribution and calendar measures. Rolling statistical features are lagged by one day; several TA-Lib indicators and OBV include the current completed session. These are end-of-day predictors and do not justify a fill at that same closing price.


In [ ]:
feature_list = []

def parkinson_vol(high, low, period=20):
    """Parkinson volatility (based on high/low) / 帕金森波动率（基于最高最低价）"""
    h_l = np.log(high / low)
    var = 1 / (4 * np.log(2)) * h_l ** 2
    return np.sqrt(var.shift(1).rolling(period).mean() * 252)

def garman_klass_vol(open_, high, low, close, period=20):
    """Garman-Klass volatility (based on OHLC) / 加曼-克拉斯波动率（基于开高低收）"""
    u = np.log(high / open_)
    d = np.log(low / open_)
    c = np.log(close / open_)
    var = 0.5 * (u - d) ** 2 - (2 * np.log(2) - 1) * c ** 2
    return np.sqrt(var.shift(1).rolling(period).mean() * 252)
df['overnight_return'] = np.log(df['open_price'] / df['pre_close_price'])
df['intraday_return'] = np.log(df['close_price_nvnf'] / df['open_price'])
feature_list.extend(['daily_return', 'overnight_return', 'intraday_return'])
for window in [5, 10, 20, 60, 120]:
    df[f'return_{window}d'] = df['daily_return'].shift(1).rolling(window).sum()
    feature_list.append(f'return_{window}d')
df['high_low_range'] = (df['high_price'] - df['low_price']) / df['close_price_nvnf']
df['open_close_range'] = abs(df['close_price_nvnf'] - df['open_price']) / df['close_price_nvnf']
feature_list.extend(['high_low_range', 'open_close_range'])
for window in [5, 10, 20, 60, 120]:
    df[f'volatility_{window}d'] = df['daily_return'].shift(1).rolling(window).std()
    feature_list.append(f'volatility_{window}d')
df['parkinson_vol'] = parkinson_vol(df['high_price'], df['low_price'], period=20)
df['garman_klass_vol'] = garman_klass_vol(df['open_price'], df['high_price'], df['low_price'], df['close_price_nvnf'], period=20)
upper, middle, lower = ta.BBANDS(df['close_price_nvnf'], timeperiod=20)
df['bb_width'] = upper - lower
feature_list.extend(['parkinson_vol', 'garman_klass_vol', 'bb_width'])
df['volume_volatility_5d'] = df['volume'].shift(1).rolling(5).std()
df['volume_volatility_20d'] = df['volume'].shift(1).rolling(20).std()
feature_list.extend(['volume_volatility_5d', 'volume_volatility_20d'])
for window in [6, 14, 28]:
    df[f'rsi_{window}d'] = ta.RSI(df['close_price_nvnf'], timeperiod=window)
    feature_list.append(f'rsi_{window}d')
df['macd_line'], df['macd_signal'], df['macd_hist'] = ta.MACD(df['close_price_nvnf'])
feature_list.extend(['macd_line', 'macd_signal', 'macd_hist'])
df['stoch_k'], df['stoch_d'] = ta.STOCH(df['high_price'], df['low_price'], df['close_price_nvnf'])
feature_list.extend(['stoch_k', 'stoch_d'])
df['williams_r'] = ta.WILLR(df['high_price'], df['low_price'], df['close_price_nvnf'], timeperiod=14)
feature_list.append('williams_r')
df['roc_10d'] = ta.ROC(df['close_price_nvnf'], timeperiod=10)
feature_list.append('roc_10d')
for window in [5, 10, 20, 60, 120]:
    df[f'ma_{window}d'] = ta.MA(df['close_price_nvnf'], timeperiod=window)
    feature_list.append(f'ma_{window}d')
df['ema_20d'] = ta.EMA(df['close_price_nvnf'], timeperiod=20)
feature_list.append('ema_20d')
df['adx_14d'] = ta.ADX(df['high_price'], df['low_price'], df['close_price_nvnf'], timeperiod=14)
feature_list.append('adx_14d')
for window in [5, 20]:
    df[f'volume_ma_{window}d'] = df['volume'].shift(1).rolling(window).mean()
    feature_list.append(f'volume_ma_{window}d')
df['volume_change'] = np.log(df['volume'] / df['volume'].shift(1))
feature_list.append('volume_change')
df['obv'] = ta.OBV(df['close_price_nvnf'], df['volume'])
df['obv_momentum_5d'] = df['obv'].diff(5)
feature_list.extend(['obv', 'obv_momentum_5d'])
df['return_skew_20d'] = df['daily_return'].shift(1).rolling(20).skew()
df['return_kurt_20d'] = df['daily_return'].shift(1).rolling(20).kurt()
df['sharpe_20d'] = df['daily_return'].shift(1).rolling(20).mean() / df['daily_return'].shift(1).rolling(20).std() * np.sqrt(252)
feature_list.extend(['return_skew_20d', 'return_kurt_20d', 'sharpe_20d'])
df['day_of_week'] = df['time'].dt.dayofweek
feature_list.append('day_of_week')
df = df.dropna().reset_index(drop=True)
print({'valid_samples': len(df), 'candidate_features': len(feature_list)})


## 4. Chronological holdout and training-only preprocessing

Keep the latest 20% as the outer test set, and discard the last training observation whose target crosses the boundary. The holdout is not used for feature fitting or parameter search. However, the historical design fits preprocessing and feature filters on the whole outer training set **before** the internal time-series folds. The internal CV estimates are therefore not a nested, fold-isolated estimate of the complete selection procedure; they may be optimistic.


In [ ]:
X = df[feature_list].to_numpy()
y = df['label'].to_numpy()
dates = df['time'].to_numpy()
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size - 1], X[train_size:]
y_train, y_test = y[:train_size - 1], y[train_size:]
dates_train, dates_test = dates[:train_size - 1], dates[train_size:]
if len(np.unique(y_train)) != 2 or len(np.unique(y_test)) != 2:
    raise ValueError('Both classes are required in train and test sets to evaluate ROC-AUC.')
tscv = TimeSeriesSplit(n_splits=5)
for train_idx, valid_idx in tscv.split(X_train):
    if len(np.unique(y_train[train_idx])) != 2 or len(np.unique(y_train[valid_idx])) != 2:
        raise ValueError('Every CV training/validation fold must contain both classes.')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
feature_names = np.asarray(feature_list)
print({'train_count': len(y_train), 'test_count': len(y_test),
       'train_start': str(dates_train[0]), 'train_end': str(dates_train[-1]),
       'test_start': str(dates_test[0]), 'test_end': str(dates_test[-1])})


## 5. Filter → wrapper → embedded feature selection

Apply the historical sequence: variance threshold, mutual information, correlation filtering, recursive feature elimination with five time-series folds, then LightGBM split-count importance. After standardization, the variance filter mostly detects constant features rather than low-variance variables in their original units. The final feature count is data-dependent, not forced to be three.


In [ ]:
var_selector = VarianceThreshold(threshold=0.001)
X_train_filtered = var_selector.fit_transform(X_train_scaled)
X_test_filtered = var_selector.transform(X_test_scaled)
filtered_names = feature_names[var_selector.get_support()]

mi_scores = mutual_info_classif(X_train_filtered, y_train, random_state=SEED)
mi_df = pd.DataFrame({'feature': filtered_names, 'score': mi_scores}).sort_values('score', ascending=False)
mi_names = mi_df.loc[mi_df['score'] >= mi_df['score'].median(), 'feature'].to_numpy()
mi_idx = [list(filtered_names).index(name) for name in mi_names]
X_train_filtered = X_train_filtered[:, mi_idx]
X_test_filtered = X_test_filtered[:, mi_idx]

corr = pd.DataFrame(X_train_filtered, columns=mi_names).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_names = [column for column in upper.columns if (upper[column] > 0.9).any()]
filter_names = np.asarray([name for name in mi_names if name not in drop_names])
corr_idx = [list(mi_names).index(name) for name in filter_names]
X_train_filter = X_train_filtered[:, corr_idx]
X_test_filter = X_test_filtered[:, corr_idx]

base_model = LGBMClassifier(random_state=SEED, class_weight='balanced', n_jobs=1, verbose=-1)
rfecv = RFECV(estimator=base_model, step=1, cv=tscv, scoring='roc_auc', n_jobs=1)
rfecv.fit(X_train_filter, y_train)
X_train_wrapper = X_train_filter[:, rfecv.support_]
X_test_wrapper = X_test_filter[:, rfecv.support_]
wrapper_names = filter_names[rfecv.support_]

importance_model = LGBMClassifier(random_state=SEED, class_weight='balanced', n_jobs=1, verbose=-1)
importance_model.fit(X_train_wrapper, y_train)
importance_df = pd.DataFrame({'feature': wrapper_names,
                              'importance': importance_model.feature_importances_}).sort_values('importance', ascending=False)
cutoff = importance_df['importance'].max() * 0.01
final_names = importance_df.loc[(importance_df['importance'] > 0) &
                                (importance_df['importance'] >= cutoff), 'feature'].to_numpy()
if len(final_names) == 0:
    raise ValueError('No non-zero-importance feature remains; inspect the data and target.')
final_idx = [list(wrapper_names).index(name) for name in final_names]
X_train_final = X_train_wrapper[:, final_idx]
X_test_final = X_test_wrapper[:, final_idx]
print({'candidate_count': len(feature_names), 'filter_count': len(filter_names),
       'wrapper_count': len(wrapper_names), 'final_features': final_names.tolist(),
       'wrapper_cv_auc': float(rfecv.cv_results_['mean_test_score'].max())})


## 6. Baseline and parameter search

The full grid has 486 configurations across six parameters and five folds. This is a substantial search on a small dataset, so the best internal CV score is a selection statistic, not proof of predictive power. The historical internal folds have no embargo/gap; the one-row outer-boundary exclusion is not applied independently inside each fold.


In [ ]:
baseline_model = LGBMClassifier(
    n_estimators=100, num_leaves=15, max_depth=4, learning_rate=0.05,
    min_child_samples=100, reg_alpha=1.0, reg_lambda=1.0,
    class_weight='balanced', random_state=SEED, n_jobs=1, verbose=-1,
)
baseline_model.fit(X_train_final, y_train)
baseline_test_auc = roc_auc_score(y_test, baseline_model.predict_proba(X_test_final)[:, 1])

full_grid = {
    'num_leaves': [10, 20, 31], 'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1], 'n_estimators': [50, 100, 200],
    'min_child_samples': [50, 100, 150], 'class_weight': [None, 'balanced'],
}
quick_grid = {
    'num_leaves': [10], 'max_depth': [3], 'learning_rate': [0.01, 0.05],
    'n_estimators': [50], 'min_child_samples': [50], 'class_weight': [None],
}
grid_search = GridSearchCV(
    LGBMClassifier(reg_alpha=1.0, reg_lambda=1.0, random_state=SEED, n_jobs=1, verbose=-1),
    param_grid=quick_grid if QUICK_MODE else full_grid,
    cv=tscv, scoring='roc_auc', n_jobs=1, error_score='raise',
)
grid_search.fit(X_train_final, y_train)
best_model = grid_search.best_estimator_
print({'search_mode': 'quick' if QUICK_MODE else 'full',
       'best_params': grid_search.best_params_,
       'internal_cv_auc': float(grid_search.best_score_)})


## 7. Holdout evaluation

As in the historical study, choose a classification threshold by training-set F1. This uses in-sample predictions, not out-of-fold predictions, and can overfit the threshold. Inspect precision and recall together; a high recall alone does not establish useful prediction. ROC-AUC is threshold-independent. No significance claim is made without an appropriate uncertainty analysis.


In [ ]:
train_probability = best_model.predict_proba(X_train_final)[:, 1]
test_probability = best_model.predict_proba(X_test_final)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, train_probability)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-10)
best_threshold = float(thresholds[np.argmax(f1_scores)])
test_prediction = (test_probability >= best_threshold).astype(int)
metrics = {
    'data_kind': DATA_SOURCE_KIND, 'search_mode': 'quick' if QUICK_MODE else 'full',
    'train_count': len(y_train), 'test_count': len(y_test),
    'candidate_count': len(feature_names), 'selected_features': final_names.tolist(),
    'baseline_test_auc': float(baseline_test_auc),
    'internal_cv_auc': float(grid_search.best_score_),
    'train_auc': float(roc_auc_score(y_train, train_probability)),
    'test_auc': float(roc_auc_score(y_test, test_probability)),
    'classification_threshold': best_threshold,
    'test_accuracy': float((test_prediction == y_test).mean()),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, test_prediction, labels=[0, 1],
                            target_names=['not above threshold', 'above 0.5%'], zero_division=0))
matrix = confusion_matrix(y_test, test_prediction, labels=[0, 1])
fpr, tpr, _ = roc_curve(y_test, test_probability)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr, label=f"Test AUC = {metrics['test_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title='ROC curve')
axes[0].legend()
axes[1].imshow(matrix, cmap='Blues')
for row in range(2):
    for column in range(2):
        axes[1].text(column, row, matrix[row, column], ha='center', va='center')
axes[1].set(xticks=[0, 1], yticks=[0, 1], xlabel='Predicted class', ylabel='Actual class', title='Confusion matrix')
fig.suptitle(f'{DATA_SOURCE_KIND} / {metrics["search_mode"]} run')
fig.tight_layout()
plt.show()


## 8. Optional local exports

Exports are disabled by default and go to ignored `outputs/private/` when enabled. Re-running the notebook may expose local data through notebook outputs; clear all cell outputs before committing or sharing. The public file intentionally contains no executed outputs.


In [ ]:
if WRITE_RESULTS:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    (RESULTS_DIR / 'classification_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    pd.DataFrame(classification_report(y_test, test_prediction, output_dict=True,
                                       zero_division=0)).T.to_csv(RESULTS_DIR / 'classification_report.csv')
    pd.DataFrame({'feature': final_names}).to_csv(RESULTS_DIR / 'selected_features.csv', index=False)
    fig.savefig(RESULTS_DIR / 'classification_evaluation.png', dpi=150, bbox_inches='tight')
    print('Classification results saved to outputs/private/.')


## Interpretation and next steps

The historical holdout AUC of 0.5302 indicates weak discrimination; it does not demonstrate statistically significant alpha. The original exploratory trading section is deliberately **excluded**: it omitted costs, mixed log-return and simple-return compounding, misaligned the benchmark and signal return periods, and used an incorrect drawdown calculation. It also assumed a same-close fill for end-of-day features and did not map the index to an executable instrument. No trading returns, Sharpe ratio or drawdown from that section are presented as validated results.

Further work would fit every transformation/feature selector inside each training fold, use nested or walk-forward evaluation with appropriate boundary gaps, select thresholds on out-of-fold or separate validation predictions, and quantify uncertainty. A trading study would require a specified tradable instrument, execution timing, aligned return periods, transaction costs, and correct portfolio accounting. These are planned improvements, not completed features of this notebook.
